In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\koush\Desktop\GenAi project folder\all_features.csv")



In [2]:
df = df.drop(columns=["file","client_ip","app","category","mode","run_id",])

In [3]:
y = (df["label"] == "genai").astype(int).values
X = df.drop(columns=["label"]).values

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [5]:
import torch

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [6]:
# Add channel dimension for CNN
X = X.unsqueeze(1)

print(X.shape)

torch.Size([106, 1, 35])


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)

torch.Size([84, 1, 35])


In [8]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.conv1 = nn.Conv1d(1, 16, kernel_size=3)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3)

        self.pool = nn.MaxPool1d(2)

        # dropout layers
        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)

        conv_output = ((num_features - 2) - 2) // 2

        self.fc1 = nn.Linear(32 * conv_output, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = self.pool(x)

        # dropout after CNN block
        x = self.dropout1(x)

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))

        # dropout before final layer
        x = self.dropout2(x)

        x = torch.sigmoid(self.fc2(x))

        return x

In [9]:
model = SimpleCNN(num_features=X.shape[2])

In [10]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [12]:
epochs = 100

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze()

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 4.2683
Epoch 2, Loss: 4.1648
Epoch 3, Loss: 4.2469
Epoch 4, Loss: 4.1945
Epoch 5, Loss: 4.1429
Epoch 6, Loss: 4.2227
Epoch 7, Loss: 4.2248
Epoch 8, Loss: 4.1858
Epoch 9, Loss: 4.1985
Epoch 10, Loss: 4.1505
Epoch 11, Loss: 4.1198
Epoch 12, Loss: 4.1031
Epoch 13, Loss: 4.0780
Epoch 14, Loss: 4.0649
Epoch 15, Loss: 3.9882
Epoch 16, Loss: 4.1357
Epoch 17, Loss: 4.0841
Epoch 18, Loss: 3.9136
Epoch 19, Loss: 3.9613
Epoch 20, Loss: 3.8575
Epoch 21, Loss: 3.9608
Epoch 22, Loss: 3.8501
Epoch 23, Loss: 3.9659
Epoch 24, Loss: 3.8688
Epoch 25, Loss: 3.8612
Epoch 26, Loss: 3.7656
Epoch 27, Loss: 3.7882
Epoch 28, Loss: 3.6072
Epoch 29, Loss: 3.8985
Epoch 30, Loss: 3.6818
Epoch 31, Loss: 3.6056
Epoch 32, Loss: 3.6464
Epoch 33, Loss: 3.7192
Epoch 34, Loss: 3.5748
Epoch 35, Loss: 3.4362
Epoch 36, Loss: 3.4275
Epoch 37, Loss: 3.5008
Epoch 38, Loss: 3.3189
Epoch 39, Loss: 3.3858
Epoch 40, Loss: 3.4554
Epoch 41, Loss: 3.3267
Epoch 42, Loss: 3.4042
Epoch 43, Loss: 3.2323
Epoch 44, Loss: 3.20

In [13]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

preds = []
true = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch).squeeze()

        predictions = (outputs > 0.5).int()

        preds.extend(predictions.tolist())
        true.extend(y_batch.int().tolist())

print(confusion_matrix(true, preds))
print(classification_report(true, preds))

[[8 3]
 [3 8]]
              precision    recall  f1-score   support

           0       0.73      0.73      0.73        11
           1       0.73      0.73      0.73        11

    accuracy                           0.73        22
   macro avg       0.73      0.73      0.73        22
weighted avg       0.73      0.73      0.73        22

